# Step 6: The final run (test chats opened once)
### Neeraj, MMAI 894 (router fine-tuning)

**What happens here, once:** the 40 locked test chats and the team's 31 fresh chats are opened for the first and only time, and every router is scored on them by the same code:

| Router | What it is |
|---|---|
| Haiku 5.5 | the team's router today (reference) |
| Sonnet 5.5 | the large prompted model |
| Jev, same information | Jev given the same rules and examples as Haiku (notebook 04b) |
| Jev, first version | for completeness |
| **Fine-tuned, safest** | full fine-tuning + attack answer: misses fewest people on the practice chats, but hands many ordinary chats to staff |
| **Fine-tuned, balanced** | LoRA + attack answer + its own unsure line: best overall of the fine-tuned options on the practice chats |

**Nothing is chosen here.** Every setting (epochs, cut-offs, unsure lines, seeds, models, repeats) was frozen in `config/final_settings.json` from the practice chats before this notebook was written. The fine-tuned models are trained on all 160 practice chats with 5 seeds; the test and fresh chats are scored only after the last epoch, and their answers are never given to training.

**What you need to do:**
1. Accelerator **GPU T4**, Internet **On**, and the `OPENROUTER_API_KEY` secret ticked for this notebook.
2. Make sure OpenRouter has at least **$5 of credit** and that the key's own limit leaves at least $5 (expected cost about $3; the run stops itself at $6).
3. Run all cells: about 30 to 40 minutes. If the session stops, run again: finished training runs and saved answers are reused.
4. Download `results_06.zip` (directly in `/kaggle/working`) and the notebook into `results/step06`.

Everything shown is computed from the runs; this text holds no results.

## Step 0: Get the latest code

In [ ]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}
!pip install -q peft
!pip uninstall -y -q torchao                 # Kaggle's older torchao makes the new peft refuse to run; we do not use it

%cd {REPO_DIR}
sys.path.insert(0, "src")
importlib.invalidate_caches()
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]

for need in ("config/final_settings.json", "src/northstar/finetune/final_run.py", "data/fresh/fresh_chats.csv"):
    assert os.path.exists(need), f"missing {need}: has everything been pushed to GitHub?"
!git log --oneline -3

## Step 1: Machine, key and the frozen settings

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:          # not on Kaggle
    UserSecretsClient = None

from northstar.finetune import baselines as B
from northstar.finetune import final_run as F
from northstar.finetune import protocol as P
from northstar.finetune import router_options as O
from northstar.finetune import routing as R

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "No GPU found. Set Accelerator to GPU T4."
API_KEY = None
try:                                    # Kaggle secret; without one, the saved answers are replayed for free
    API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    pass

S = F.load_settings()
print("Settings frozen on", S["frozen_on"], "from", S["frozen_from"])
for name, s in S["fine_tuned"].items():
    print(f"  {name:9s} {s['label']}: input {s['input']}, epochs {s['training']['epochs']}, cut-offs escalation {s['escalation_cutoff']} / "
          f"attack {s['attack_cutoff']}, unsure line {s['unsure_line']}")
print("  language models:", {m: v["repeats"] for m, v in S["language_models"].items()}, "| inputs:", S["language_model_inputs"])

RESULTS = "/kaggle/working/results_06"
RUNS, PREDS, CACHE = f"{RESULTS}/fine_tuned", f"{RESULTS}/preds", f"{RESULTS}/cache"
os.makedirs(RESULTS, exist_ok=True)
shutil.copy("config/final_settings.json", f"{RESULTS}/final_settings.json")
client = B.Client(API_KEY, CACHE, budget_usd=S["api_budget_usd"])
if os.path.exists("results/step06/cache"):            # the answers saved in the repo: reused, never paid for twice
    shutil.copytree("results/step06/cache", CACHE, dirs_exist_ok=True)
print("API key:", "loaded" if API_KEY else "none, so the saved answers are replayed (free; stops at any call that was not saved)")
pool = P.pool(P.load_all())

## Step 2: Open the test and fresh chats (once)
Both sets are locked in the code; opening them needs `final=True` and an environment switch, and each opening is written to `outputs/test_access.log`.

In [ ]:
os.environ["NORTHSTAR_ALLOW_TEST"] = "1"
test = P.test_set(P.load_all(), final=True, note="final run, notebook 06, settings frozen in config/final_settings.json")
fresh = P.fresh_set(final=True, note="final run, notebook 06, settings frozen in config/final_settings.json")
held = pd.concat([test.assign(dataset="test"), fresh.assign(dataset="fresh")], ignore_index=True)
for name, d in (("test", test), ("fresh", fresh)):
    print(f"{name:5s}: {len(d)} chats, {int(d['escalation_label'].sum())} need a person, {int(d['attack_label'].sum())} attacks, "
          f"{int((d['language'] == 'fr').sum())} French, {d['customer_id'].nunique()} customers")
assert not set(held["id"]) & set(pool["id"]), "a held-out chat is also in the practice pool"
print(open("outputs/test_access.log").read())

## Step 3: Smoke test (practice chats only)
One seed, one epoch, scoring 8 practice chats: proves the training code runs before the real training. No held-out chat is used here.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
for name, s in S["fine_tuned"].items():
    cfg = F.train_config({**s["training"], "epochs": 1})
    t0 = time.time()
    F.train_and_score(cfg, s["input"], pool, pool.head(8).assign(dataset="smoke"), [1], f"/kaggle/working/smoke/{name}",
                      tokenizer=tokenizer, verbose=False)
    print(f"{name:9s} smoke test OK: {time.time() - t0:.0f}s for 1 epoch")

## Step 4: Train both fine-tuned setups on all 160 practice chats, score the held-out chats once
Five seeds each. The LoRA setup's trained add-ons and heads are saved (a few MB per seed) for the hand-over; the full fine-tuning weights (1.1 GB each) are not. One model of each setup is kept to measure its speed on a CPU (2 cores, one chat at a time, like the team's server).

In [ ]:
SEEDS = S["fine_tuned_common"]["seeds"]
paths, latency = {}, {}
first_messages = list(P.input_text(test, "first_message"))
for name, s in S["fine_tuned"].items():
    print(f"=== {name}: {s['label']}")
    cfg = F.train_config(s["training"])
    paths[name], model = F.train_and_score(cfg, s["input"], pool, held, SEEDS, f"{RUNS}/{name}", keep_seed=SEEDS[0],
                                           save_trained_weights=cfg.lora, tokenizer=tokenizer)
    latency[name] = F.cpu_latency(model, tokenizer, first_messages, cfg.max_length, threads=2)
    print(f"  CPU speed, one model: median {latency[name]['median_s']:.3f}s, p90 {latency[name]['p90_s']:.3f}s per chat")
    del model
    torch.cuda.empty_cache()
pd.DataFrame(latency).T.to_csv(f"{RESULTS}/fine_tuned_cpu_latency.csv")

## Step 5: The language models on the same chats
Same prompt, settings and repeats as on the practice chats. A probe of 3 chats per model first, with a cost estimate, then the full run. Saved answers are reused if the cell is run again.

In [ ]:
LLMS = S["language_models"]
INPUTS = S["language_model_inputs"]
probe = pd.concat([B.run_variant(client, m, held, "first_message", 0, PREDS, limit=3) for m in LLMS], ignore_index=True)
assert not probe["fallback"].any(), "a probe call gave no usable answer: see the error column"
per_chat = probe.groupby("model")["cost_usd"].mean()
estimate = sum(per_chat[m] * len(held) * len(INPUTS) * v["repeats"] for m, v in LLMS.items()) * 1.15
print("cost per chat:", per_chat.round(6).to_dict(), f"| estimated: ${estimate:.2f} (limit ${S['api_budget_usd']:.0f})")
assert estimate < S["api_budget_usd"], "estimate above the limit"

for m, v in LLMS.items():
    for inp in INPUTS:
        for rep in range(v["repeats"]):
            d = B.run_variant(client, m, held, inp, rep, PREDS, workers=3)
            print(f"{m:6s} {inp:17s} repeat {rep}: fallbacks {int(d['fallback'].sum())} | ${d['cost_usd'].sum():.3f}", flush=True)
print(f"New money spent in this session: ${client.spent:.2f}")

## Step 6: The final scores
For each set (test, fresh) and input, every router scored the same way: safety first (missed, right team, attacks to the right team), then efficiency (extra handoffs, kept with the wrong desk), then the strict handled-right with its 95% range (resampling customers), and an exact paired test against Haiku on the same chats. The fine-tuned routers read the first message; their row is compared with Haiku on the first message.

In [ ]:
NAMES = {"haiku": "Haiku (team's router)", "sonnet": "Sonnet", "jev2": "Jev, same information as Haiku", "jev": "Jev, first version",
         "safest": "Fine-tuned, safest", "balanced": "Fine-tuned, balanced"}
LAT = pd.read_csv("results/step04/latency.csv").set_index("model")["median_ms"] / 1000
LAT = pd.concat([LAT, pd.read_csv("results/step04b/latency.csv").set_index("model")["median_ms"] / 1000])
rows, routed, pairs, spread = [], {}, [], []
for dname in ("test", "fresh"):
    mask = (held["dataset"] == dname).values
    d = held[mask].reset_index(drop=True)
    for inp in INPUTS:
        for m in LLMS:
            reps = [r[mask].reset_index(drop=True) for r in B.load_preds(PREDS, m, inp, held)]
            llm = O.combine_llm(reps)
            lang = np.where(np.mean([(r["language"] == "fr").values for r in reps], axis=0) > 0.5, "fr", "en")
            row, rt = F.score_router(d, O.option_llm_alone(llm), lang)
            rows.append(dict(dataset=dname, input=inp, router=NAMES[m], seconds=LAT.get(m, np.nan), **row))
            routed[(dname, inp, m)] = rt["handled_right"].values
    for name, s in S["fine_tuned"].items():
        combined, per_seed, seeds, secs = F.tuned_answers(paths[name], held, s["escalation_cutoff"], s["attack_cutoff"])
        lang = P.input_text(d, s["input"]).map(R.language_rule).values
        row, rt = F.score_router(d, O.option_attack_head(combined[mask].reset_index(drop=True)), lang, s["unsure_line"])
        rows.append(dict(dataset=dname, input=s["input"], router=NAMES[name], seconds=latency[name]["median_s"], **row))
        routed[(dname, s["input"], name)] = rt["handled_right"].values
        for seed, a in zip(seeds, per_seed):
            r1, _ = F.score_router(d, O.option_attack_head(a[mask].reset_index(drop=True)), lang, s["unsure_line"], n_boot=10)
            spread.append(dict(dataset=dname, router=NAMES[name], seed=seed, type_accuracy=r1["type_accuracy"],
                               handled_right=r1["handled_right"], missed=r1["missed"], extra_handoffs=r1["extra_handoffs"]))
    for (dn, inp, key), hr in routed.items():
        if dn == dname and key != "haiku":
            pairs.append(dict(dataset=dname, input=inp, router=NAMES[key], versus="Haiku", **F.paired_vs(routed[(dname, inp, "haiku")], hr)))

final = pd.DataFrame(rows)
final.to_csv(f"{RESULTS}/final_scores.csv", index=False)
pd.DataFrame(pairs).to_csv(f"{RESULTS}/final_paired_vs_haiku.csv", index=False)
spread = pd.DataFrame(spread)
spread.to_csv(f"{RESULTS}/final_fine_tuned_by_seed.csv", index=False)

show = final.copy()
show["type right %"] = (100 * show["type_accuracy"]).round(1)
show["handled right % [95% range]"] = [f"{100 * h:.1f} [{100 * lo:.0f}-{100 * hi:.0f}]" for h, lo, hi in zip(show["handled_right"], show["handled_ci_low"], show["handled_ci_high"])]
show["needs a person: missed"] = show["missed"].astype(str) + " of " + show["needs_person"].astype(str)
show["attacks to right team"] = show["attacks_right_team"].astype(str) + " of " + show["attacks"].astype(str)
cols = ["router", "type right %", "needs a person: missed", "right_team", "attacks to right team", "extra_handoffs", "kept_wrong_desk",
        "handled right % [95% range]", "usd_per_1000", "seconds"]
for (dn, inp), g in show.groupby(["dataset", "input"], sort=False):
    print(f"\n===== {dn} chats, {inp.replace('_', ' ')}")
    display(g.sort_values("handled_right", ascending=False)[cols].set_index("router"))
display(pd.DataFrame(pairs).round(4))

## Step 7: How steady are the fine-tuned models, and did the practice chats predict the test?
Each of the 5 seeds scored on its own (the combined router averages them), and the practice-chat estimate frozen in the settings next to the test and fresh results.

In [ ]:
display(spread.groupby(["dataset", "router"])[["type_accuracy", "handled_right", "missed", "extra_handoffs"]].agg(["mean", "std"]).round(3))
check = []
for name, s in S["fine_tuned"].items():
    e = s["pool_estimate"]
    check.append(dict(router=NAMES[name], where="practice chats (estimate)", handled_right=e["handled_right"], missed=e["missed"], extra_handoffs=e["extra_handoffs"]))
    for dn in ("test", "fresh"):
        r = final[(final["router"] == NAMES[name]) & (final["dataset"] == dn)].iloc[0]
        check.append(dict(router=NAMES[name], where=f"{dn} chats", handled_right=round(r["handled_right"], 3), missed=r["missed"], extra_handoffs=r["extra_handoffs"]))
pd.DataFrame(check)

## Step 8: Recordings for the team's end-to-end test
The team's harness can replay a router's recorded answers (`RecordedRouter` in `evaluation/setups.py`). These files hold each fine-tuned router's answers on the test and fresh chats in that format: type, escalation, reason (attack -> adversarial, else from the type), language (word rule) and a confidence rescaled so that the team's 0.55 unsure line acts as the router's own line.

In [ ]:
os.makedirs(f"{RESULTS}/recordings", exist_ok=True)
for name, s in S["fine_tuned"].items():
    combined, _, _, _ = F.tuned_answers(paths[name], held, s["escalation_cutoff"], s["attack_cutoff"])
    lang = P.input_text(held, s["input"]).map(R.language_rule).values
    rec = F.recording(held, combined, lang, s["unsure_line"], 1000 * latency[name]["median_s"], tokenizer)
    rec.to_csv(f"{RESULTS}/recordings/router_finetuned-{name}_test_and_fresh.csv", index=False)
    print(name, "recording:", len(rec), "chats")

## Step 9: Save the results
Download `results_06.zip` and the notebook (File -> Download Notebook) into `results/step06`.

In [ ]:
shutil.copy("outputs/test_access.log", f"{RESULTS}/test_access.log")
shutil.make_archive("/kaggle/working/results_06", "zip", RESULTS)
print(f"results_06.zip {os.path.getsize('/kaggle/working/results_06.zip') / 1e6:.1f} MB")
for name in sorted(os.listdir(RESULTS)):
    print("  ", name)

## What this run shows, and its limits
- **Shows:** the honest, once-only comparison of every router on chats nobody tuned on: the 40 test chats (same generator as the training data) and the 31 fresh chats (another model family, reviewed by two people, labelled under the team's final policy).
- **Limits:**
  - Small sets: 40 and 31 chats, so the 95% ranges are wide; differences of a few chats are not proof.
  - The fresh chats follow the team's final policy, which is stricter on some escalations than the data the models learned from; every router meets that difference, the fine-tuned ones most directly.
  - The team's own Haiku and Sonnet test runs (single runs, full conversation) used the same prompt; this run repeats them on both inputs with the same code as every other router.
  - The fine-tuned speed is one model on 2 CPU cores; the combined five-seed router takes about five times as long unless one seed is used.
  - The router alone is scored here; the team's whole-agent test (graded replies) is the next step and uses the recordings saved above.